In [ ]:
!pip3 install torch torchvision --index-url https://download.pytorch.org/whl/cu126

Looking in indexes: https://download.pytorch.org/whl/cu126


In [ ]:
#---------------------------
# MODEL INFERENCE
# (YOLO is pretrained, if something of a new domain needs to be identified, then model has to be retrained)
#---------------------------

!uv pip install torch torchvision

try:
    import ultralytics
except ImportError:
    !pip install -q ultralytics

print("Ultralytics working")

from ultralytics import YOLO
print("YOLO imported")

Using Python 3.12.12 environment at: /usr
Audited 2 packages in 211ms
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 28.9 MB/s eta 0:00:00
Ultralytics working
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
YOLO imported


In [ ]:
!pip install opencv-python  # or opencv-python-headless, opencv-contrib-python, etc.
!pip install -U albumentationsx

In [ ]:
#-----------------------------
# AUGMENTATION
#-----------------------------

import torch
from torchvision.transforms import v2
from torchvision import tv_tensors

img = torch.randint(0, 256, size=(3, H, W), dtype=torch.uint8)
boxes = torch.randint(0, H // 2, size=(3, 4))
boxes[:, 2:] += boxes[:, :2]
boxes = tv_tensors.BoundingBoxes(boxes, format="XYXY", canvas_size=(H, W))

# The same transforms can be used!
img, boxes = transforms(img, boxes)
# And you can pass arbitrary input structures
output_dict = transforms({"image": img, "boxes": boxes})

In [ ]:
import cv2
cv2.setNumThreads(0)
# Optionally, disable OpenCL if not needed or causing issues
# cv2.ocl.setUseOpenCL(False)

import torch
from torchvision.io import decode_image

def load_image_torchvision(path):
    with open(path, 'rb') as f:
        image_bytes = f.read()
    image = decode_image(torch.frombuffer(image_bytes, dtype=torch.uint8))  # Returns tensor in RGB
    return image.permute(1, 2, 0).numpy()  # Convert to numpy if needed


In [ ]:
import albumentations as A
import numpy as np

# OBB in cxcywh format (center, width, height, angle in degrees)
bbox_params = A.BboxParams(
    coord_format='cxcywh',
    bbox_type='obb',
    label_fields=['labels'],
)


# Example pipeline
fast = A.Compose([
    A.RandomCrop(224, 224, p=1.0), # Example random crop
    A.HorizontalFlip(p=0.5),
    A.RandomBrightnessContrast(p=0.2),
], bbox_params=A.BboxParams(coord_format='yolo', # Specify input format
                           label_fields=['class_labels'] # Specify label argument name(s)
                           ))


#**Important: Transform Compatibility**: Not all transforms support bounding boxes.
# Before building your pipeline, check the [Supported Targets by Transform](/docs/reference/supported-targets-by-transform) reference
#to ensure all your chosen transforms are compatible with bboxes. For example, [`RandomBrightnessContrast`]
#(https://explore.albumentations.ai/transform/RandomBrightnessContrast) only affects image pixels, while
#[`HorizontalFlip`](https://explore.albumentations.ai/transform/HorizontalFlip) affects both images and bboxes.



In [ ]:
from ultralytics import YOLO
import torch

torch.cuda.empty_cache()
gc.collect()


model = YOLO("yolov8n-obb.pt")

# YOLO's built-in augmentation is VERY powerful
results = model.train(
    data="DOTAv1.yaml",  # Your existing YAML - no changes needed!
    epochs=100,
    imgsz=640,
    batch=32,
    device=0,

    # Heavy augmentation for DOTA (aerial images)
    degrees=20.0,          # ±20° rotation
    translate=0.1,         # 10% translation
    scale=0.5,             # ±50% scaling
    shear=5.0,             # Shear transform
    flipud=0.5,            # Vertical flip
    fliplr=0.5,            # Horizontal flip
    mosaic=1.0,            # Mosaic augmentation
    mixup=0.1,             # MixUp
    hsv_h=0.015,           # Hue variation
    hsv_s=0.7,             # Saturation variation
    hsv_v=0.4,             # Brightness variation
    copy_paste=0.1,        # Copy-paste augmentation
    auto_augment='randaugment',

    amp=True,
    cache="ram",
    workers=8,
    patience=15,
    project="runs/obb",
    name="dota_heavy_aug"
)

print("✅ Training complete with heavy augmentation!")

In [ ]:
import os
import shutil
from google.colab import files

# Folder to zip
folder_to_zip = "/content/runs/obb/val"

# Zip the folder
output_filename = "obb_validation"
archive_name = shutil.make_archive(output_filename, 'zip', folder_to_zip)
print(f"Folder '{folder_to_zip}' has been zipped to '{archive_name}'.")

# Download the zipped file
files.download(archive_name)
print(f"'{archive_name}' is being downloaded.")

Folder '/content/runs/obb/val' has been zipped to '/content/obb_validation.zip'.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

'/content/obb_validation.zip' is being downloaded.


In [ ]:
#---------------------------
# VALIDATION
#---------------------------

# Load a model
model = YOLO("yolo26n-obb.pt")  # load an official model

# Validate the model
metrics = model.val(data="DOTAv1.yaml")  # no arguments needed, dataset and settings remembered
metrics.box.map  # map50-95(B)
metrics.box.map50  # map50(B)
metrics.box.map75  # map75(B)
metrics.box.maps  # a list containing mAP50-95(B) for each category

Ultralytics 8.4.21 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO26n-obb summary (fused): 132 layers, 2,449,332 parameters, 0 gradients, 5.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2170.0±923.1 MB/s, size: 1672.4 KB)
val: Scanning /content/datasets/DOTAv1/labels/val.cache... 458 images, 2 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 458/458 192.1Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 29/29 1.1s/it 33.2s
                   all        458      28853      0.761      0.546      0.628      0.493
                 plane         70       2531      0.905      0.731      0.821      0.684
                  ship        108       8960       0.86      0.589      0.717      0.553
          storage tank         55       2888      0.842      0.383      0.555      0.399
      baseball diamond         53        214      0.759      0.679      0.753      0.589
          tennis court         9

array([    0.68403,     0.55325,     0.39867,     0.58854,     0.91346,     0.52114,     0.44254,     0.53686,     0.14355,     0.68499,     0.48429,     0.41387,     0.26927,     0.42248,     0.33148])

In [ ]:
#---------------------------
# PREDICTION
#---------------------------

# Load a model
model = YOLO("yolo26n-obb.pt")  # load an official model

# Predict with the model
results = model("/content/Bird's Eye Drone of Shipping Containers.mp4")  # predict on an image

# Access the results
for result in results:
    xywhr = result.obb.xywhr  # center-x, center-y, width, height, angle (radians)
    xyxyxyxy = result.obb.xyxyxyxy  # polygon format with 4-points
    names = [result.names[cls.item()] for cls in result.obb.cls.int()]  # class name of each box
    confs = result.obb.conf  # confidence score of each box


image 1/1 /content/boats.jpg: 576x1024 170 ships, 3 harbors, 66.5ms
Speed: 4.6ms preprocess, 66.5ms inference, 0.5ms postprocess per image at shape (1, 3, 576, 1024)


In [ ]:
!pip install supervision

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.9/41.9 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 217.4/217.4 kB 9.0 MB/s eta 0:00:00


In [ ]:
import numpy as np
import supervision as sv
from ultralytics import YOLO


model = YOLO("/content/best (1).pt")
box_annotator = sv.BoxAnnotator()

def callback(frame: np.ndarray, _: int) -> np.ndarray:
    results = model(frame)[0]
    detections = sv.Detections.from_ultralytics(results)
    # no tracker.update_with_detections(...)
    return box_annotator.annotate(frame.copy(), detections=detections)

sv.process_video(
    source_path="/content/Bird's Eye Drone of Shipping Containers.mp4",
    target_path="/content/Bird's Eye Drone of Shipping Containers_OBB.mp4",
    callback=callback
)

#Save per-frame detections

frames_generator = sv.get_video_frames_generator("/content/Bird's Eye Drone of Shipping Containers.mp4")

with sv.CSVSink("detections.csv") as sink:
    for frame_index, frame in enumerate(frames_generator):
        results = model(frame)[0]
        detections = sv.Detections.from_ultralytics(results)
        sink.append(detections, {"frame_index": frame_index})

Streaming output truncated to the last 5000 lines.
0: 384x640 (no detections), 13.4ms
Speed: 6.1ms preprocess, 13.4ms inference, 0.5ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 12.8ms
Speed: 3.0ms preprocess, 12.8ms inference, 0.5ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 15.5ms
Speed: 2.7ms preprocess, 15.5ms inference, 0.6ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 17.3ms
Speed: 3.2ms preprocess, 17.3ms inference, 0.6ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 16.5ms
Speed: 1.9ms preprocess, 16.5ms inference, 0.6ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 18.5ms
Speed: 3.0ms preprocess, 18.5ms inference, 0.6ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 14.7ms
Speed: 3.6ms preprocess, 14.7ms inference, 0.5ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no det

In [ ]:
from google.colab.patches import cv2_imshow
import os, cv2
import numpy as np

vid_path = "/content/ABODA/video2.avi"
base_name = os.path.splitext(os.path.basename(vid_path))[0]

cap = cv2.VideoCapture(vid_path)
backSub = cv2.createBackgroundSubtractorMOG2()

if not cap.isOpened():
    print("Error opening video file")
else:
    fourcc = cv2.VideoWriter_fourcc(*'XVID')
    fps = cap.get(cv2.CAP_PROP_FPS)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    out = cv2.VideoWriter(f"{base_name}_output.avi", fourcc, fps, (width, height))

    while cap.isOpened():

        ret, frame = cap.read()
        if not ret:
            break

        # 1️⃣ Background subtraction
        fg_mask = backSub.apply(frame)

        # 2️⃣ Remove shadows
        _, mask_thresh = cv2.threshold(fg_mask, 180, 255, cv2.THRESH_BINARY)

        # 3️⃣ Morphological opening (noise removal)
        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
        mask_clean = cv2.morphologyEx(mask_thresh, cv2.MORPH_OPEN, kernel)

        # 4️⃣ Find contours on CLEAN mask (important!)
        contours, _ = cv2.findContours(mask_clean, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

        # 5️⃣ Filter small contours
        min_contour_area = 500
        large_contours = [cnt for cnt in contours if cv2.contourArea(cnt) > min_contour_area]

        frame_out = frame.copy()

        for cnt in contours:
          if cv2.contourArea(cnt) > 500:
            x, y, w, h = cv2.boundingRect(cnt)
            cv2.rectangle(frame_out, (x, y), (x+w, y+h), (0, 0, 255), 2)

            out.write(frame_out)

    cap.release()
    out.release()

